In [1]:
# === Imports pour l'optimisation du modèle ===

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle

# scikit-learn
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Configuration
pd.set_option('display.max_columns', None)
DATA_CLEAN = '../data/clean/'

# Chargement du dataset enrichi
df = pd.read_csv(DATA_CLEAN + 'imdb_4000_enriched.csv')

print(f"Dataset chargé : {df.shape[0]} films")
print(f"Colonne director_name présente : {'director_name' in df.columns}")
print(f"Films avec réalisateur connu : {df['director_name'].notna().sum()}")

Dataset chargé : 4000 films
Colonne director_name présente : True
Films avec réalisateur connu : 4000


In [2]:
# === Préparation des features (avec ajout du réalisateur) ===

# 1. Encodage des genres (comme semaine 5)
tous_genres = pd.concat([df['genre_1'], df['genre_2'], df['genre_3']]).dropna().unique()
for genre in tous_genres:
    df[f'is_{genre}'] = (
        (df['genre_1'] == genre) | 
        (df['genre_2'] == genre) | 
        (df['genre_3'] == genre)
    ).astype(int)

genres_cols = [col for col in df.columns if col.startswith('is_')]
print(f"✅ {len(genres_cols)} genres encodés")

# 2. NOUVEAU : Top 50 réalisateurs (au-delà, ce sera trop dilué)
top_realisateurs = df['director_name'].value_counts().head(50).index.tolist()
print(f"\n✅ Top 50 réalisateurs identifiés")
print(f"   Exemples : {top_realisateurs[:5]}")

# Encodage des réalisateurs : une colonne binaire par réal du top 50
for real in top_realisateurs:
    nom_safe = real.replace(' ', '_').replace('.', '').replace("'", "")
    df[f'dir_{nom_safe}'] = (df['director_name'] == real).astype(int)

real_cols = [col for col in df.columns if col.startswith('dir_')]
print(f"✅ {len(real_cols)} colonnes réalisateurs créées")

# 3. Liste finale des features
features_num = ['imdb_score', 'duration', 'title_year', 'num_voted_users']
features_all = features_num + genres_cols + real_cols

print(f"\n📊 Total features : {len(features_all)}")
print(f"   • Numériques : {len(features_num)}")
print(f"   • Genres : {len(genres_cols)}")
print(f"   • Réalisateurs : {len(real_cols)}")

✅ 23 genres encodés

✅ Top 50 réalisateurs identifiés
   Exemples : ['Steven Spielberg', 'Woody Allen', 'Martin Scorsese', 'Clint Eastwood', 'Ridley Scott']
✅ 50 colonnes réalisateurs créées

📊 Total features : 77
   • Numériques : 4
   • Genres : 23
   • Réalisateurs : 50


In [3]:
# === Construction et normalisation de la matrice X ===

X = df[features_all].copy()

# Remplir les valeurs manquantes
for col in features_num:
    X[col] = X[col].fillna(X[col].median())

# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"✅ Matrice X_scaled : {X_scaled.shape}")
print(f"   {X_scaled.shape[0]} films × {X_scaled.shape[1]} features")

✅ Matrice X_scaled : (4000, 77)
   4000 films × 77 features


In [4]:
# === Construction de 3 modèles avec métriques différentes ===

# Modèle 1 : Euclidean (celui de la semaine 5)
modele_euclidean = NearestNeighbors(n_neighbors=6, metric='euclidean')
modele_euclidean.fit(X_scaled)

# Modèle 2 : Cosine Similarity (suggéré par ton formateur)
modele_cosine = NearestNeighbors(n_neighbors=6, metric='cosine')
modele_cosine.fit(X_scaled)

# Modèle 3 : Manhattan
modele_manhattan = NearestNeighbors(n_neighbors=6, metric='manhattan')
modele_manhattan.fit(X_scaled)

print("✅ 3 modèles entraînés avec succès")
print("   • Modèle Euclidean (référence)")
print("   • Modèle Cosine Similarity")
print("   • Modèle Manhattan")

✅ 3 modèles entraînés avec succès
   • Modèle Euclidean (référence)
   • Modèle Cosine Similarity
   • Modèle Manhattan


In [5]:
# === Fonction de comparaison des 3 modèles ===

def comparer_modeles(titre_film, n=5):
    """Compare les recommandations des 3 métriques côte à côte"""
    
    masque = df['movie_title'].str.strip().str.lower() == titre_film.lower().strip()
    if not masque.any():
        print(f"❌ Film '{titre_film}' non trouvé")
        return
    
    idx = df[masque].index[0]
    info = df.iloc[idx]
    
    print(f"\n{'='*80}")
    print(f"🎬 FILM DE RÉFÉRENCE : {info['movie_title'].strip()} ({info['title_year']})")
    print(f"   Note : {info['imdb_score']} | Réalisateur : {info['director_name']}")
    genres = [info[g] for g in ['genre_1', 'genre_2', 'genre_3'] if pd.notna(info[g])]
    print(f"   Genres : {', '.join(genres)}")
    print(f"{'='*80}\n")
    
    # Récupération des recommandations pour chaque modèle
    modeles = {
        'EUCLIDEAN': modele_euclidean,
        'COSINE': modele_cosine,
        'MANHATTAN': modele_manhattan
    }
    
    resultats = {}
    for nom, modele in modeles.items():
        distances, indices = modele.kneighbors([X_scaled[idx]])
        recos = []
        for i, idx_v in enumerate(indices[0][1:]):
            v = df.iloc[idx_v]
            recos.append(f"{i+1}. {v['movie_title'].strip()[:35]:<35} ({v['title_year']}) - {v['imdb_score']}")
        resultats[nom] = recos
    
    # Affichage côte à côte
    print(f"{'EUCLIDEAN':<50} | {'COSINE':<50} | {'MANHATTAN':<50}")
    print("-" * 156)
    for i in range(n):
        print(f"{resultats['EUCLIDEAN'][i]:<50} | {resultats['COSINE'][i]:<50} | {resultats['MANHATTAN'][i]:<50}")
    print()


# Test sur 4 films représentatifs de styles différents
comparer_modeles("Inception")
comparer_modeles("The Dark Knight")
comparer_modeles("The Lion King")
comparer_modeles("Forrest Gump")


🎬 FILM DE RÉFÉRENCE : Inception (2010)
   Note : 8.8 | Réalisateur : Christopher Nolan
   Genres : Action, Adventure, Sci-Fi

EUCLIDEAN                                          | COSINE                                             | MANHATTAN                                         
------------------------------------------------------------------------------------------------------------------------------------------------------------
1. The Matrix                          (1999) - 8.7 | 1. The Avengers                        (2012) - 8.1 | 1. The Avengers                        (2012) - 8.1
2. The Avengers                        (2012) - 8.1 | 2. The Matrix                          (1999) - 8.7 | 2. The Matrix                          (1999) - 8.7
3. Interstellar                        (2014) - 8.6 | 3. Iron Man                            (2008) - 7.9 | 3. Iron Man                            (2008) - 7.9
4. Batman Begins                       (2005) - 8.3 | 4. Batman Begins         

In [6]:
# === Modèle final retenu : Cosine Similarity avec 75 features ===

# Le modèle Cosine est notre champion
modele_final = modele_cosine

# Fonction de recommandation finale (version optimisée)
def recommander_films_v2(titre_film, n=5):
    """
    Recommande n films similaires en utilisant le modèle optimisé
    (Cosine Similarity + features incluant les réalisateurs).
    """
    masque = df['movie_title'].str.strip().str.lower() == titre_film.lower().strip()
    
    if not masque.any():
        print(f"❌ Film '{titre_film}' non trouvé dans le dataset")
        return None
    
    idx = df[masque].index[0]
    distances, indices = modele_final.kneighbors([X_scaled[idx]], n_neighbors=n+1)
    
    recos = []
    for idx_voisin, dist in zip(indices[0][1:], distances[0][1:]):
        v = df.iloc[idx_voisin]
        genres = ' | '.join([str(v[g]) for g in ['genre_1', 'genre_2', 'genre_3'] if pd.notna(v[g])])
        recos.append({
            'titre': v['movie_title'].strip(),
            'année': int(v['title_year']),
            'note': v['imdb_score'],
            'réalisateur': v['director_name'],
            'genres': genres,
            'similarité': round(1 - dist, 3)  # cosine : on convertit la distance en similarité
        })
    
    return pd.DataFrame(recos)


# Test final
print("🎬 Test final sur 'Pulp Fiction' :\n")
print(recommander_films_v2("Pulp Fiction"))

🎬 Test final sur 'Pulp Fiction' :

                      titre  année  note         réalisateur  \
0  The Shawshank Redemption   1994   9.3      Frank Darabont   
1           The Dark Knight   2008   9.0   Christopher Nolan   
2           American Beauty   1999   8.4          Sam Mendes   
3        American History X   1998   8.6           Tony Kaye   
4               City of God   2002   8.7  Fernando Meirelles   

                   genres  similarité  
0           Crime | Drama       0.978  
1  Action | Crime | Drama       0.956  
2                   Drama       0.916  
3           Crime | Drama       0.901  
4           Crime | Drama       0.897  


In [7]:
# === Sauvegarde du modèle optimisé ===

import os

os.makedirs('../models', exist_ok=True)

elements_optimises = {
    'modele': modele_final,
    'scaler': scaler,
    'X_scaled': X_scaled,
    'df': df,
    'features': features_all,
    'metadata': {
        'metric': 'cosine',
        'n_features': len(features_all),
        'n_films': len(df),
        'version': '2.0_optimisee'
    }
}

with open('../models/modele_reco.pkl', 'wb') as f:
    pickle.dump(elements_optimises, f)

taille_mb = os.path.getsize('../models/modele_reco.pkl') / (1024 * 1024)
print(f"✅ Modèle optimisé sauvegardé !")
print(f"   Fichier : ../models/modele_reco.pkl")
print(f"   Taille : {taille_mb:.2f} MB")
print(f"\n📊 Métadonnées :")
for cle, val in elements_optimises['metadata'].items():
    print(f"   • {cle} : {val}")

✅ Modèle optimisé sauvegardé !
   Fichier : ../models/modele_reco.pkl
   Taille : 10.20 MB

📊 Métadonnées :
   • metric : cosine
   • n_features : 77
   • n_films : 4000
   • version : 2.0_optimisee


In [8]:
!pip install streamlit

In [9]:
import streamlit as st
print(f"Streamlit version : {st.__version__}")

Streamlit version : 1.51.0
